# DressApp Eyes — Vision Smoke Test

Diagnostic notebook for the self-hosted Gemma-4 E2B Eyes service.
**Goal:** confirm whether the model can actually see a photo, with no DressApp backend in the loop.

## What this notebook does
1. Calls your Eyes container's `POST /predict` directly with a real image.
2. Shows raw output, finish reason, token counts, and total latency — so we can tell the difference between *slow*, *empty*, and *blind*.
3. Re-runs the same call with a *blank canvas* image to prove the model's output actually depends on pixel input (the canonical "is the projector wired?" check).
4. Optionally calls Gemini 2.5 Flash on the same image so you can score Gemma quality against the Google baseline.

## Before you run
Your Eyes container is internal-only on the docker network (`eyes:7860`). For the notebook to reach it, expose it temporarily one of three ways — pick whichever you prefer:

**Option A (simplest, ~2 min) — open the port directly**
```bash
ssh root@178.104.114.210
cd /srv/AI-Stylist/deploy
# Edit docker-compose.yml: change `expose: - "7860"` under the eyes
# service to `ports: - "7860:7860"`. Then:
docker compose up -d eyes
ufw allow 7860/tcp           # if ufw is enabled
```
Eyes is now reachable at `http://178.104.114.210:7860`. **Bearer auth is still enforced**, so it's safe — only your `EYES_API_TOKEN` holders can hit `/predict`.

**Option B — Caddy subpath**
Add this block to `Caddyfile` inside the `dressapp.co` site block:
```
@eyes_debug path /_eyes_debug/*
handle @eyes_debug {
    uri strip_prefix /_eyes_debug
    reverse_proxy eyes:7860
}
```
Then `docker compose restart caddy`. Eyes is reachable at `https://dressapp.co/_eyes_debug` (TLS handled).

**Option C — cloudflared / ngrok one-shot**
```bash
ssh root@178.104.114.210
docker run --rm --network dressapp_dress -p 4040:4040 \
    cloudflare/cloudflared tunnel --url http://eyes:7860
```
Copy the `*.trycloudflare.com` URL it prints.

When you're done testing, **remember to revert** Option A (re-tighten to `expose:`) or pull down the Caddy block from Option B — the bearer auth is good but minimising attack surface is better.

## 1 · Setup

In [ ]:
%pip install -q requests pillow ipywidgets
import base64, io, json, time, textwrap, requests
from PIL import Image
from IPython.display import display, JSON, Markdown, Image as IPImage
from google.colab import files  # type: ignore
print('ready')

## 2 · Config — fill these in

`EYES_URL` is whatever you set up in the prep step (e.g. `http://178.104.114.210:7860`, `https://dressapp.co/_eyes_debug`, or a `*.trycloudflare.com` URL).

`EYES_API_TOKEN` is the bearer token from `/srv/AI-Stylist/deploy/.env` on your VPS:
```bash
ssh root@178.104.114.210 'grep EYES_API_TOKEN /srv/AI-Stylist/deploy/.env'
```

In [ ]:
EYES_URL       = 'http://178.104.114.210:7860'        #@param {type:'string'}
EYES_API_TOKEN = 'paste-your-EYES_API_TOKEN-here'      #@param {type:'string'}
REQUEST_TIMEOUT_S = 180                                 #@param {type:'integer'}

# Optional — for the Gemini comparison cell at the bottom. Leave
# blank to skip that section entirely.
GEMINI_API_KEY = ''                                    #@param {type:'string'}

EYES_URL = EYES_URL.rstrip('/')
headers = {'Authorization': f'Bearer {EYES_API_TOKEN}', 'Content-Type': 'application/json'}

# Smoke-test connectivity + capability flags.
r = requests.get(f'{EYES_URL}/healthz', headers=headers, timeout=10)
print('healthz:', r.status_code, r.json() if r.ok else r.text[:300])
r = requests.get(f'{EYES_URL}/', timeout=10)
if r.ok:
    info = r.json()
    print('vision_enabled :', info.get('vision_enabled'))
    print('engine         :', info.get('engine'))
    print('model          :', info.get('model'))
    print('arch metadata  :', info.get('gguf_metadata'))

If `vision_enabled` is **`False`**, the smoke test cannot prove vision works — fix the deploy first (likely `EYES_MMPROJ_FILE` not propagated, or the eyes container needs `--force-recreate`).

If `vision_enabled` is **`True`**, continue.

## 3 · Upload a photo

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise SystemExit('No file uploaded')
filename, raw_bytes = next(iter(uploaded.items()))

img = Image.open(io.BytesIO(raw_bytes)).convert('RGB')
# Match the backend's pre-processing: 1280px long side, JPEG q=82.
img.thumbnail((1280, 1280))
buf = io.BytesIO(); img.save(buf, format='JPEG', quality=82, optimize=True)
image_b64 = base64.b64encode(buf.getvalue()).decode('ascii')
print(f'{filename} → {len(buf.getvalue())/1024:.1f} KB JPEG')
display(IPImage(data=buf.getvalue(), width=300))

## 4 · The actual test — call Eyes on the photo

We use the **same prompt the DressApp backend sends** (lifted verbatim from `garment_vision.py`) so the result here is exactly what would land in your closet.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent('''\
    You are The Eyes — DressApp\u2019s visual garment analyst. You look at a single clothing photo and describe it in exhaustive, merchandisable detail. Your output is used to auto-fill an Add-Item form that a user will review, so be confident but never invent sensitive claims (e.g. do not guess a specific brand unless clearly visible; leave brand blank otherwise).

    Return ONLY a JSON object with the following shape (all keys optional except `title`):
    {
      "name": string,
      "title": string,
      "caption": string,
      "category": "Top"|"Bottom"|"Outerwear"|"Full Body"|"Footwear"|"Accessories"|"Underwear",
      "sub_category": string,
      "item_type": string,
      "brand": string|null,
      "gender": "men"|"women"|"unisex"|"kids",
      "dress_code": "casual"|"smart-casual"|"business"|"formal"|"athletic"|"loungewear",
      "season": string[],
      "colors": [{"name": string, "pct": integer 0..100}, ...],
      "fabric_materials": [{"name": string, "pct": integer 0..100}, ...],
      "pattern": "solid"|"striped"|"plaid"|"floral"|"herringbone"|"polka"|"paisley"|"geometric"|"abstract",
      "state": "new"|"used",
      "condition": "bad"|"fair"|"good"|"excellent",
      "quality": "budget"|"mid"|"premium"|"luxury",
      "size": string|null,
      "tags": string[]
    }
''').strip()

USER_TEXT = 'Analyse this garment photograph and return the JSON object only — no commentary.'

def call_eyes(image_b64: str, *, max_tokens: int = 2400, temperature: float = 0.1) -> dict:
    payload = {
        'system': SYSTEM_PROMPT,
        'prompt': USER_TEXT,
        'image_b64': image_b64,
        'image_mime': 'image/jpeg',
        'max_tokens': max_tokens,
        'temperature': temperature,
        'json_mode': True,
    }
    t0 = time.perf_counter()
    r = requests.post(f'{EYES_URL}/predict', headers=headers, json=payload, timeout=REQUEST_TIMEOUT_S)
    dt = time.perf_counter() - t0
    return {'status': r.status_code, 'elapsed_s': round(dt, 1), **(r.json() if r.ok else {'error': r.text[:600]})}

result = call_eyes(image_b64)
print('HTTP                :', result.get('status'))
print('Total latency       :', result.get('elapsed_s'), 's')
print('Server elapsed_ms   :', result.get('elapsed_ms'))
print('Vision used         :', result.get('vision_used'))
print('Vision disabled     :', result.get('vision_disabled'))
print('Tokens prompt       :', result.get('tokens_prompt'))
print('Tokens completion   :', result.get('tokens_completion'))
print('Finish reason       :', result.get('finish_reason'))
print('-' * 50)
raw_output = result.get('output', '') or result.get('error', '')
print('Raw model output:')
print(raw_output[:4000])

### Read the output above — what each row tells you

| Row | Healthy value | Failure mode it diagnoses |
|---|---|---|
| `Vision used` | `True` | If `False`, mmproj didn't load — fix the deploy. |
| `Finish reason` | `stop` (or `length` only on huge outputs) | `length` on a small JSON means thinking ate all tokens — `--reasoning-budget 0` didn't take effect. |
| `Tokens completion` | 200–600 for a clean JSON | `≥1000` with empty `output` ⇒ model is rambling/thinking. |
| `Total latency` | 8–25 s on CPX32 with thinking off | 60 s+ ⇒ thinking still on. |
| Raw output | Starts with `{`, contains real garment terms | Empty / parroting filenames / wrong category ⇒ fine-tune quality issue. |

## 5 · Try to parse the JSON

Mirrors the backend's `_extract_json` — finds the largest `{...}` block and json-loads it. Survives mild commentary around the JSON.

In [ ]:
import re
def extract_json(s: str):
    if not s:
        return None
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', s, flags=re.S)
    if m:
        try: return json.loads(m.group(1))
        except Exception: pass
    a, b = s.find('{'), s.rfind('}')
    if a != -1 and b > a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    try: return json.loads(s)
    except Exception: return None

parsed = extract_json(raw_output)
if parsed:
    display(JSON(parsed, expanded=True))
else:
    print('Could not parse JSON from model output. The fine-tune is not producing schema-conformant output for this image.')

## 6 · Vision-blindness control test

**This is the cleanest proof the model is actually seeing pixels.** We send the SAME prompt with a *blank grey canvas* in place of the real image. If the projector is wired and Gemma is sighted, the output for the blank should be markedly different from the output for your real photo (or the model should refuse / produce a generic placeholder).

If both calls return identical / similarly-confident garment descriptions, **the model is hallucinating the entire response from the prompt alone** — a strong signal that the vision projector isn't actually connected to the LLM during inference.

In [ ]:
blank = Image.new('RGB', (768, 768), (180, 180, 180))
buf2 = io.BytesIO(); blank.save(buf2, format='JPEG', quality=82)
blank_b64 = base64.b64encode(buf2.getvalue()).decode('ascii')

control = call_eyes(blank_b64)
print('Blank-canvas latency :', control.get('elapsed_s'), 's')
print('Blank-canvas tokens  :', control.get('tokens_completion'))
print('Raw blank output:')
print((control.get('output') or '')[:2000])
print('-' * 50)
print('PARSED:')
blank_parsed = extract_json(control.get('output') or '')
display(JSON(blank_parsed, expanded=False) if blank_parsed else Markdown('*(unparseable)*'))

## 7 · (Optional) Compare against Gemini 2.5 Flash on the same photo

Provide a `GEMINI_API_KEY` at the top to run this. Skips itself if blank.

In [ ]:
if not GEMINI_API_KEY.strip():
    print('GEMINI_API_KEY not set — skipping comparison.')
else:
    %pip install -q google-generativeai
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-2.5-flash', system_instruction=SYSTEM_PROMPT)
    t0 = time.perf_counter()
    resp = model.generate_content([
        {'mime_type': 'image/jpeg', 'data': base64.b64decode(image_b64)},
        USER_TEXT,
    ])
    dt = time.perf_counter() - t0
    print(f'Gemini latency: {dt:.1f}s')
    gemini_text = resp.text
    print(gemini_text[:2000])
    gemini_parsed = extract_json(gemini_text)
    if gemini_parsed:
        print('-' * 50)
        print('GEMINI PARSED:')
        display(JSON(gemini_parsed, expanded=False))

## 8 · Side-by-side scoreboard

What you should see for a known garment (e.g. that black leather belt):

| Field | Gemma (your fine-tune) | Gemini 2.5 Flash | Truth |
|---|---|---|---|
| `category` | ? | `Accessories` | Accessories |
| `sub_category` | ? | `Belt` | Belt |
| `colors[0].name` | ? | black / brown / silver | matches the photo |
| `pattern` | ? | `solid` | solid |

**Decision tree based on this notebook's results:**

* `vision_enabled=False` → fix infra; can't evaluate model.
* `vision_enabled=True` AND blank-canvas output ≈ photo output → projector not wired into the inference path despite metadata saying it is. Likely a `--mmproj` flag mismatch or the gguf was loaded by an older llama.cpp build. Re-pull `llama.cpp` HEAD.
* `vision_enabled=True` AND blank-canvas output is clearly generic but real photo output is *wrong* (e.g. "shirt" for a belt) → vision works but the fine-tune is undertrained for those classes. Add more belt/accessory examples to your dataset.
* `vision_enabled=True` AND output is roughly accurate → ship it 🎉. Compare quality vs Gemini and decide whether the latency / privacy / cost trade-off is worth it for production.